### Exercise 1 - Entropy of Neural Responses

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from entropy_utils import compute_entropy,normalize_distribution

In [2]:
# Simulate Spike Count Distribution

spike_counts = np.array([0, 1, 2])
probabilities = np.array([0.5, 0.25, 0.25])

H = compute_entropy(probabilities)

print("Entropy H(R):", H, "bits")

Entropy H(R): 1.5 bits


### Task

Change distribution to:

[0.95, 0.05]
Observe entropy change.

In [3]:
# Changed distribution to [0.95, 0.05]
probabilities_new = np.array([0.95, 0.05])

H_new = compute_entropy(probabilities_new)

print("Original entropy H(R):", H, "bits")
print("New entropy H(R):", H_new, "bits")

Original entropy H(R): 1.5 bits
New entropy H(R): 0.28639695711595625 bits


**Q1 Answer:**
- Original [0.5, 0.25, 0.25] -> H = 1.5 bits
- New [0.95, 0.05] -> H = 0.286 bits

The entropy dropped a lot. A skewed distribution carries **less** uncertainty, because the neuron almost always gives the same response, so its output is very predictable.

### Exercise 2 - Mutual Information

In [4]:
# Simulate Stimulus-Response Mapping
np.random.seed(0)

stimulus = np.random.choice([0, 1], size=1000)

response = stimulus + np.random.normal(0, 0.5, size=1000)
response = (response > 0.5).astype(int)

In [5]:
# Compute MI
from mutual_information import compute_mutual_information

I = compute_mutual_information(stimulus, response)
print("Mutual Information:", I, "bits")

Mutual Information: 0.4263533917403184 bits


### Task
Increase noise.
Observe MI change.
Explain why.

In [6]:
# Increase noise (standard deviation) from 0.5 to 2.0
np.random.seed(0)

stimulus = np.random.choice([0, 1], size=1000)

response_noisy = stimulus + np.random.normal(0, 2.0, size=1000)
response_noisy = (response_noisy > 0.5).astype(int)

I_noisy = compute_mutual_information(stimulus, response_noisy)
print("Mutual Information (noise = 0.5):", I, "bits")
print("Mutual Information (noise = 2.0):", I_noisy, "bits")

Mutual Information (noise = 0.5): 0.4263533917403184 bits
Mutual Information (noise = 2.0): 0.05166961321271846 bits


**Q2 Answer:**
- Noise 0.5 -> MI = 0.426 bits
- Noise 2.0 -> MI = 0.052 bits

MI drops a lot when noise increases. With more noise, the responses for stimulus 0 and stimulus 1 overlap, so the response often points to the wrong stimulus. The response becomes mostly random and depends less on the stimulus, so seeing the response does not reduce our uncertainty about the stimulus very much.

### Exercise 3 - Spike Timing Information

In [7]:
#Create two spike trains:
train_A = [10, 20, 30]
train_B = [70, 80, 90]

In [8]:
from spike_train_analysis import compute_spike_train_entropy

### Compare entropy using:

Bin size = 100 ms
Bin size = 10 ms

Explain difference.

In [9]:
spike_trains = [train_A, train_B]
duration = 100   # ms

H_100 = compute_spike_train_entropy(spike_trains, duration, bin_size=100)
H_10 = compute_spike_train_entropy(spike_trains, duration, bin_size=10)

print("Entropy with 100 ms bins:", H_100, "bits")
print("Entropy with 10 ms bins:", H_10, "bits")

Entropy with 100 ms bins: 0.0 bits
Entropy with 10 ms bins: 1.0 bits


**Q3 Answer:**
- 100 ms bins -> both trains become [3] (same pattern) -> H = 0 bits
- 10 ms bins -> the two trains give different patterns -> H = 1 bit

Smaller bins keep the spike timing, so there are more possible patterns and the entropy is higher. With very large bins only the spike count is kept, so trains with different timing look the same and the timing information is lost.

### Exercise 4 - Coding Efficiency

In [10]:
from coding_efficiency import compare_distributions

compare_distributions()

Uniform distribution: [0.25 0.25 0.25 0.25] -> Entropy = 2.0 bits
Skewed distribution : [0.7 0.1 0.1 0.1] -> Entropy = 1.3568 bits
Maximum possible entropy (4 states) = log2(4) = 2.0 bits
Uniform distribution has higher entropy, so it is more efficient.


(np.float64(2.0), np.float64(1.3567796494470397))

##### Which distribution is more efficient?
##### Why does uniform maximize entropy?

**Q4 Answer:**
- Uniform -> H = 2.0 bits (the maximum, log2(4))
- Skewed -> H = 1.357 bits

The **uniform** distribution is more efficient. When all firing rates are used equally, no response is more predictable than another, so each response carries the most information. The neuron uses its whole range of firing rates instead of wasting most of the time on one rate.

Uniform maximizes entropy because every outcome is equally likely, so the next response is the hardest to guess (most uncertainty).

### Critical Thinking Questions
##### Can a neuron have high entropy but low mutual information? Explain.
##### Why does noise reduce mutual information?
##### How does bin size affect spike train entropy?
##### Does higher firing rate imply higher information?

**Critical Thinking Answers:**

1. **Yes.** Entropy only measures how variable the response is, while MI measures how much of that variability is caused by the stimulus. A neuron that fires randomly has high entropy but low MI. For example, with noise = 2.0, H(R) is about 1.0 bit but MI is only about 0.05 bits.

2. Noise adds randomness that has nothing to do with the stimulus. This increases the uncertainty about the response even when the stimulus is known (H(R|S)), and since MI = H(R) - H(R|S), the MI goes down.

3. Smaller bins capture precise spike timing, so there are more possible patterns and the entropy is higher. Larger bins only count spikes, giving fewer patterns and lower entropy (100 ms gave 0 bits, 10 ms gave 1 bit).

4. **No.** Information depends on how reliably the response changes with the stimulus, not on how many spikes there are. A neuron that always fires at a high rate is predictable and carries no information, and a high rate that is noisy carries little information.